# Decode data

Each month's reshaped parquet stores one row per declaration item, with every column still a raw string. Several columns are AFIP codes that need to be resolved against the catalogs in `docs/`.

Column -> catalog mapping (`CATALOGS` in `argentina_afip_comex/pipeline.py`):

| Column | Catalog | Notes |
|---|---|---|
| `ADU` | `ANEXOIV_Aduanas.csv` | Aduana de registro |
| `M` | `ANEXOV_Vias_y_Medios_de_Transporte.csv` | Medio de transporte |
| `UN` | `ANEXOXV_Unidades_de_medida.csv` | Unidad de medida (uses `Descripcion Ampliada`, since `Descripcion` is inconsistent/abbreviated) |
| `DIV` | `ANEXOXI_Divisas.csv` | Divisa |
| `PAI` (1st) | `ANEXOVII_Paises.csv` | Renamed to `PAIS_ORIGEN` |
| `PAI_duplicated_0` (2nd) | `ANEXOVII_Paises.csv` | Renamed to `PAIS_PROCEDENCIA` |

`COD` (tax concept) is not decoded here: 0.3 already summed `MONTO` across tax concepts and dropped `COD`, so there's nothing left to resolve against `ANEXOXX_Conceptos_Recaudatorios.csv`.

Numeric columns (`CANT_UNIDAD_MEDIDA`, `FOB_DOLAR`, `FOB_TOTAL`) are cast from padded strings to `Float64`; empty strings become null. `MONTO_TRIBUTADO_TOTAL` was already cast and summed in 0.3. Codes missing from a catalog also become null.

Each month is decoded into `data/processed/months/importaciones_<yyyymm>_decoded.parquet` (months already decoded are skipped), and all months are then combined into the final `data/processed/importaciones_decoded.parquet`.

In [ ]:
import logging

import polars as pl
from tqdm.auto import tqdm

from argentina_afip_comex.pipeline import INTERIM_DIR, combine_months, decode_month
from argentina_afip_comex.query import DATA_PATH

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

reshaped_paths = sorted(INTERIM_DIR.rglob("impo_*_reshaped.parquet"))
decoded_paths = [
    decode_month(reshaped_path)
    for reshaped_path in tqdm(reshaped_paths, desc="Decoding months")
]
combine_months(decoded_paths, extend_existing=False)
logger.info(f"Wrote decoded dataset to {DATA_PATH}")

In [ ]:
# Preview the written output, plus how many values per decoded column ended up
# null (blank codes or codes missing from the catalogs).
decoded = pl.scan_parquet(DATA_PATH)
print(decoded.head().collect())
print(
    decoded.select(
        [
            "ADUANA",
            "MEDIO_TRANSPORTE",
            "UNIDAD_MEDIDA",
            "DIVISA",
            "PAIS_ORIGEN",
            "PAIS_PROCEDENCIA",
        ]
    )
    .null_count()
    .collect()
)